# Lennard-Jones fluid: molecular dynamics and the radial distribution function

*physchem · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/physchem/lj_liquid_rdf)

Five hundred atoms interacting through the Lennard-Jones potential in a periodic box, integrated with velocity Verlet. All pair vectors live in one broadcast tensor with the minimum-image convention, and three state points (gas, liquid near the triple point and a compressed solid) run as one batch. A Berendsen thermostat sets the temperature, an energy-conserving NVE run follows, and histograms of pair distances give $g(r)$: flat for the gas, damped shells for the liquid, sharp lattice peaks for the solid. The virial gives the pressure.

**Vector form**

$$m\,\ddot{\mathbf r}_i = \sum_{j\ne i} 24\varepsilon\Big[2\Big(\frac{\sigma}{r_{ij}}\Big)^{12} - \Big(\frac{\sigma}{r_{ij}}\Big)^{6}\Big]\frac{\mathbf r_{ij}}{r_{ij}^2},\qquad P = \rho k_BT + \frac{1}{3V}\Big\langle\sum_{i<j}\mathbf r_{ij}\cdot\mathbf F_{ij}\Big\rangle,\qquad g(r) = \frac{\langle n(r)\rangle}{\rho\,4\pi r^2\,\Delta r}$$

**Index (tensor) form**

$$d_{saij} = x_{sia} - x_{sja} - L_s\,\mathrm{round}\!\Big(\frac{x_{sia} - x_{sja}}{L_s}\Big) \qquad r^2_{sij} = d_{saij}d_{saij} \qquad F_{sia} = 24\,\big(2r_{sij}^{-14} - r_{sij}^{-8}\big)\,d_{saij} \qquad W_s = \tfrac12\,r^2_{sij}\,f_{sij}/r_{sij}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $\rho_s,\ T_s$ | `state_points(rho_liquid, T_liquid)` | (S,) | gas $(0.05, 1.5)$, liquid $(0.84, 0.72)$, solid $(1.10, 0.60)$ in reduced units |
| $x_{sia},\ L_s$ | `fcc_lattice(n_cells, rho)` | (S, N, 3), (S,) | start on an fcc lattice, $N = 4n^3$ atoms, box $L_s = (N/\rho_s)^{1/3}$ |
| $d_{saij}$ | `xt[:, :, :, None] - xt[:, :, None, :]` | (S, 3, N, N) | every pair vector at once by broadcasting, then wrapped to the nearest image |
| $r^2_{sij} = d_{saij}d_{saij}$ | `(d * d).sum(1)` | (S, N, N) | repeated $a$ ⇒ summed |
| $f_{sij}/r_{sij}$ | `24 * inv2 * (2 * inv12 - inv6)` | (S, N, N) | $-u^{\prime}(r)/r$, zero on the diagonal and beyond $r_c = 2.5\sigma$ |
| $F_{sia}$ | `(f_over_r[:, None] * d).sum(-1)` | (S, N, 3) | repeated $j$ ⇒ summed; $d_{saji} = -d_{saij}$ gives Newton's third law for free |
| $\text{velocity Verlet}$ | `verlet(x, v, L, dt, n_steps)` | (S, N, 3) | half kick, drift (wrapped into the box), new forces, half kick: time-reversible and energy-conserving |
| $v \to v\sqrt{1 + \tfrac{\Delta t}{\tau_T}(T_0/T - 1)}$ | `verlet(..., T_target=T)` | (S, N, 3) | Berendsen thermostat during equilibration only |
| $P_s$ | `virial_pressure(r2, f_over_r, T, rho, L)` | (S,) | $\rho T + W/3V$ plus the tail correction for pairs beyond $r_c$ |
| $g_s(r_k)$ | `radial_distribution(pair_histogram(r2, ...))` | (S, K) | pairs per shell divided by the ideal-gas count $\tfrac12 N\rho\,4\pi r^2\Delta r$ |

Reduced units: lengths in $\sigma$, energies in $\varepsilon$, mass 1, $T^* = k_BT/\varepsilon$, time $\tau = \sigma\sqrt{m/\varepsilon}$; for argon $\sigma = 3.4$ Å, $\varepsilon/k_B = 120$ K, so the liquid state point is about 86 K. This is the simulation Rahman ran for liquid argon in 1964 (Phys. Rev. 136, A405), with the integrator and the $g(r)$ analysis Verlet introduced in 1967 (Phys. Rev. 159, 98). The potential is cut at $r_c = 2.5\sigma$ and shifted to zero there so the energy is continuous; the pressure gets the standard tail correction $\tfrac{16}{3}\pi\rho^2[\tfrac23 r_c^{-9} - r_c^{-3}]$, which assumes $g = 1$ beyond the cutoff. The Berendsen thermostat rescales velocities towards the target temperature; it does not produce exact canonical fluctuations, so it is only used to equilibrate, and the measurements come from a plain NVE run, where the flat energy trace checks the integrator. The gas and liquid start hot so the lattice melts. The first peak of the liquid $g(r)$ sits near the potential minimum $2^{1/6}\sigma$, later shells are damped and $g \to 1$ by about $3\sigma$; the solid keeps the fcc peaks. For the gas the pressure is compared with the ideal gas and with the second-virial correction $P = \rho T(1 + B_2\rho)$, $B_2$ integrated numerically. Runs are short (a few $\tau$), so averages carry statistical noise of a few per cent. The full pair tensor costs $O(N^2)$ per step; production codes use cell or neighbour lists to reach millions of atoms.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/physchem/lj_liquid_rdf/main.py
"""Main loop: fcc lattices → thermostatted equilibration → NVE production → g(r), pressure and energy drift."""

from pathlib import Path

import torch

from problems._common.runtime import cli, device, fast_mode

from . import plot, solve, tensors
from .equation import R_CUT, second_virial


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    rho, T, T_start = tensors.state_points(params["rho_liquid"], params["T_liquid"], dev)  # (S,) each
    n_cells = 4 if fast_mode() else params["n_cells"]  # n_cells ≥ 4 keeps L/2 > r_c in the solid
    x, L = tensors.fcc_lattice(n_cells, rho)  # (S, N, 3), (S,)
    S, N, _ = x.shape
    v = tensors.maxwell_velocities(x.shape, T_start, dev)
    dt = params["dt"]
    n_eq = 60 if fast_mode() else params["equilibration_steps"]
    n_prod = 60 if fast_mode() else params["production_steps"]
    print(f"{S} state points × N = {N} atoms; pair tensor d_saij {(S, 3, N, N)}; r_c = {R_CUT} σ")
    for s, name in enumerate(tensors.NAMES):
        print(f"  {name:6s} ρ* = {rho[s]:.3f}  T* = {T[s]:.2f}  box L = {L[s]:.2f} σ")

    r_max = min(0.5 * L.min().item(), 4.0)
    threads = torch.get_num_threads()
    torch.set_num_threads(1)  # one thread per small step is steadier than many fighting over a busy CPU
    try:
        x, v = solve.equilibrate(x, v, L, T, T_start, dt, n_eq)
        res = solve.production(x, v, L, rho, dt, n_prod, r_max)
    finally:
        torch.set_num_threads(threads)
    print(f"equilibration {n_eq} + NVE production {n_prod} steps, dt = {dt} τ ({(n_eq + n_prod) * dt:.0f} τ in all)")

    E, Tk, P = res["E"], res["T"], res["P"]
    peak = solve.first_peak(res["r"], res["g"])
    tail = res["r"] > 0.8 * r_max
    print("  state    ⟨T*⟩   ⟨P*⟩     energy drift / atom   g(r) peak at   ⟨g⟩ for r > {:.1f} σ".format(0.8 * r_max))
    for s, name in enumerate(tensors.NAMES):
        drift = (E[-1, s] - E[0, s]).item()
        print(
            f"  {name:6s}  {Tk[:, s].mean():.3f}  {P[:, s].mean():+.3f}   {drift:+.2e} ε"
            f"           {peak[s]:.3f} σ        {res['g'][s, tail].mean():.3f}"
        )
    print(f"LJ minimum 2^(1/6) σ = {2 ** (1 / 6):.3f} σ")

    # low density: compare with the ideal gas and the second-virial correction Z = 1 + B₂ρ
    T_gas, rho_gas = Tk[:, 0].mean().item(), rho[0].item()
    Z = P[:, 0].mean().item() / (rho_gas * T_gas)
    B2 = second_virial(T_gas)
    print(
        f"gas: P/(ρT) = {Z:.3f}   ideal gas 1   with B₂(T*={T_gas:.2f}) = {B2:.3f} σ³: 1 + B₂ρ = {1 + B2 * rho_gas:.3f}"
    )
    return plot.animate(res, L, rho, T, dt, out_dir / "lj_liquid_rdf.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/physchem/lj_liquid_rdf/equation.py
r"""Physics → tensor translation: minimum-image pair vectors, Lennard-Jones forces, virial pressure and g(r).

Indices: s state point, i j atoms, a Cartesian axis, k histogram bin.

    d_saij = x_sia - x_sja - L_s round((x_sia - x_sja) / L_s)              minimum image
    r²_sij = d_saij d_saij                                                 (a summed)
    u(r) = 4 (r⁻¹² - r⁻⁶) - u(r_c)  for r < r_c                            truncated and shifted
    F_sia = Σ_j 24 (2 r⁻¹⁴ - r⁻⁸)_sij d_saij                                (j summed)

The pair tensor is stored axis-first, (S, 3, N, N): the innermost loop then runs over j in contiguous memory,
which on a CPU is several times faster than (S, N, N, 3). For the same reason the two sums that run every step
are written as (d * d).sum(1) and (f * d).sum(-1); for these shapes einsum picks a slower path.
    P_s = ρ_s T_s + (1/3V_s) ½ Σ_ij r_sij f(r_sij) + P_tail                virial theorem
    g_s(r_k) = ⟨pairs in shell k⟩ / (½ N ρ_s 4π r_k² Δr)                   radial distribution function
"""

import math

import torch

R_CUT = 2.5  # σ


def minimum_image(x: torch.Tensor, L: torch.Tensor) -> torch.Tensor:
    """Pair vectors d_saij = x_sia - x_sja wrapped to the nearest image.   x (S, N, 3), L (S,) → (S, 3, N, N)"""
    xt = x.transpose(1, 2).contiguous()  # (S, 3, N), contiguous so d is laid out (S, 3, N, N) in memory too
    d = xt[:, :, :, None] - xt[:, :, None, :]
    box = L[:, None, None, None]
    return d - box * torch.round(d / box)


def lj_pairs(d: torch.Tensor):
    """r²_sij, pair energies u_sij and f_sij/r_sij for r < r_c (zero on the diagonal and beyond the cutoff)."""
    r2 = (d * d).sum(1)  # (S, N, N), a summed
    inside = (r2 < R_CUT**2) & (r2 > 0)  # r² = 0 only on the diagonal i = j
    inv2 = torch.where(inside, r2.reciprocal(), 0.0)  # 1/r², 0 where no interaction
    inv6 = inv2 * inv2 * inv2
    inv12 = inv6 * inv6
    u_cut = 4 * (R_CUT**-12 - R_CUT**-6)
    u = torch.where(inside, 4 * (inv12 - inv6) - u_cut, 0.0)
    f_over_r = 24 * inv2 * (2 * inv12 - inv6)  # -du/dr / r
    return r2, u, f_over_r


def forces(d: torch.Tensor, f_over_r: torch.Tensor) -> torch.Tensor:
    """F_sia = Σ_j (f/r)_sij d_saij   (j summed): Newton's third law is built in, since d_saji = -d_saij.   (S, N, 3)"""
    return (f_over_r[:, None] * d).sum(-1).transpose(1, 2)


def potential_energy(u: torch.Tensor) -> torch.Tensor:
    """U_s = ½ Σ_ij u_sij (every pair appears twice in the full matrix).   (S,)"""
    return 0.5 * u.sum((1, 2))


def kinetic_temperature(v: torch.Tensor) -> torch.Tensor:
    """T_s = Σ_ia v²_sia / (3N - 3), momentum conserved.   (S,)"""
    return torch.einsum("sia,sia->s", v, v) / (3 * v.shape[1] - 3)


def virial_pressure(r2: torch.Tensor, f_over_r: torch.Tensor, T: torch.Tensor, rho: torch.Tensor, L: torch.Tensor):
    """P_s = ρT + W/3V + P_tail, W = ½ Σ_ij r·F = ½ Σ_ij (f/r) r²; the tail adds the pairs beyond r_c (g ≈ 1 there)."""
    W = 0.5 * torch.einsum("sij,sij->s", f_over_r, r2)
    tail = 16 / 3 * math.pi * rho**2 * (2 / 3 * R_CUT**-9 - R_CUT**-3)
    return rho * T + W / (3 * L**3) + tail


def pair_histogram(r2: torch.Tensor, r_max: float, n_bins: int) -> torch.Tensor:
    """Count every pair i < j into n_bins shells of [0, r_max) for each state point.   (S, K)"""
    S, N, _ = r2.shape
    i, j = torch.triu_indices(N, N, offset=1, device=r2.device)
    r = torch.sqrt(r2[:, i, j])  # (S, P) each pair once
    k = (r / r_max * n_bins).long()
    keep = k < n_bins
    flat = (torch.arange(S, device=r2.device)[:, None] * n_bins + k)[keep]
    return (
        torch.zeros(S * n_bins, device=r2.device)
        .index_add_(0, flat, torch.ones_like(flat, dtype=r.dtype))
        .reshape(S, n_bins)
    )


def radial_distribution(counts: torch.Tensor, n_samples: int, N: int, rho: torch.Tensor, r_max: float):
    """g_s(r_k) = counts / (samples · ½ N ρ_s · shell volume_k): 1 for an ideal gas.   (S, K), centres (K,)"""
    K = counts.shape[1]
    edges = torch.linspace(0, r_max, K + 1, device=counts.device)
    shell = 4 / 3 * math.pi * (edges[1:] ** 3 - edges[:-1] ** 3)  # (K,)
    ideal = n_samples * 0.5 * N * rho[:, None] * shell[None, :]
    return counts / ideal, 0.5 * (edges[1:] + edges[:-1])


def second_virial(T: float, n: int = 20000) -> float:
    """B₂(T) = -2π ∫ (e^{-u(r)/T} - 1) r² dr for the full LJ potential, by the trapezoid rule."""
    r = torch.linspace(0.3, 30.0, n, dtype=torch.float64)
    u = 4 * (r**-12 - r**-6)
    core = 2 * math.pi * 0.3**3 / 3  # r < 0.3: e^{-u/T} = 0, the integrand is -r²
    return (-2 * math.pi * torch.trapezoid((torch.exp(-u / T) - 1) * r**2, r)).item() + core


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/physchem/lj_liquid_rdf/tensors.py
"""Build the concrete tensors: state points, an fcc starting lattice in a periodic box, Maxwell–Boltzmann velocities.

Reduced Lennard-Jones units: lengths in σ, energies in ε, mass m = 1, temperature T* = k_BT/ε, density ρ* = Nσ³/V,
time τ = σ √(m/ε). For argon σ = 3.4 Å, ε/k_B = 120 K and τ = 2.2 ps.
"""

import torch

NAMES = ("gas", "liquid", "solid")


def state_points(rho_liquid: float, T_liquid: float, device: torch.device):
    """ρ*_s, T*_s, and a hotter start temperature (S,) for each of the three state points, all run as one batch.

    gas: well above the critical temperature (T*_c ≈ 1.3) at low density; liquid: near the triple point;
    solid: compressed fcc crystal. The gas and liquid start hot so the lattice melts quickly.
    """
    rho = torch.tensor([0.05, rho_liquid, 1.10], device=device)
    T = torch.tensor([1.5, T_liquid, 0.60], device=device)
    T_start = torch.tensor([1.5, 1.5, 0.60], device=device)
    return rho, T, T_start


def fcc_lattice(n_cells: int, rho: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """x (S, N, 3) on an fcc lattice of n_cells³ cubic cells (N = 4 n_cells³) and box lengths L_s = (N/ρ_s)^{1/3}."""
    basis = torch.tensor([[0, 0, 0], [0.5, 0.5, 0], [0.5, 0, 0.5], [0, 0.5, 0.5]], device=rho.device)
    r = torch.arange(n_cells, dtype=torch.float32, device=rho.device)
    cells = torch.cartesian_prod(r, r, r)  # (n³, 3)
    unit = (cells[:, None, :] + basis[None, :, :] + 0.25).reshape(-1, 3) / n_cells  # (N, 3) in [0, 1)
    L = (unit.shape[0] / rho) ** (1 / 3)
    return unit[None] * L[:, None, None], L


def maxwell_velocities(shape: tuple, T: torch.Tensor, device: torch.device, seed: int = 0) -> torch.Tensor:
    """v (S, N, 3): Gaussian velocities with zero total momentum, scaled to exactly T_s."""
    gen = torch.Generator().manual_seed(seed)
    v = torch.randn(shape, generator=gen).to(device)
    v = v - v.mean(1, keepdim=True)
    n_dof = 3 * shape[1] - 3
    T_now = (v * v).sum((1, 2)) / n_dof
    return v * torch.sqrt(T / T_now)[:, None, None]


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/physchem/lj_liquid_rdf/solve.py
"""Velocity Verlet for all state points at once: Berendsen-thermostatted equilibration, then an NVE production run."""

import torch

from problems._common.progress import steps

from . import equation as eq


def evaluate(x: torch.Tensor, L: torch.Tensor):
    """Forces (S, N, 3), potential energy (S,), r² (S, N, N) and f/r (S, N, N) for the current positions."""
    d = eq.minimum_image(x, L)
    r2, u, f_over_r = eq.lj_pairs(d)
    return eq.forces(d, f_over_r), eq.potential_energy(u), r2, f_over_r


def verlet(x, v, L, dt, n_steps, T_target=None, tau_T=0.1, record=None, label="MD steps"):
    """Velocity Verlet: v += ½ dt F;  x += dt v;  F(x);  v += ½ dt F   (mass 1).

    With T_target (S,), a Berendsen thermostat rescales v by √(1 + dt/τ_T (T_target/T - 1)) after each step.
    record(step, x, v, U, r2, f_over_r) is called after every step (for measurements). Returns x, v.
    """
    f, U, r2, f_over_r = evaluate(x, L)
    for step in steps(n_steps, label):
        v = v + 0.5 * dt * f
        x = torch.remainder(x + dt * v, L[:, None, None])  # wrap back into the box
        f, U, r2, f_over_r = evaluate(x, L)
        v = v + 0.5 * dt * f
        if T_target is not None:
            T = eq.kinetic_temperature(v)
            v = v * torch.sqrt(1 + dt / tau_T * (T_target / T - 1)).clamp(0.8, 1.25)[:, None, None]
        if record is not None:
            record(step, x, v, U, r2, f_over_r)
    return x, v


def equilibrate(x, v, L, T, T_start, dt, n_steps):
    """First third at T_start (melts the lattice of the fluids), then at the target T, both with the thermostat."""
    hot = n_steps // 3
    x, v = verlet(x, v, L, dt, hot, T_target=T_start, label="melting")
    return verlet(x, v, L, dt, n_steps - hot, T_target=T, label="equilibration")


def production(x, v, L, rho, dt, n_steps, r_max, n_bins=120, sample_every=10, n_frames=60):
    """NVE run: total energy, temperature and virial pressure every step; g(r) and frames every few steps."""
    S, N, _ = x.shape
    out = {"E": [], "T": [], "P": [], "frames": [], "frame_step": [], "g_running": []}
    counts = torch.zeros(S, n_bins, device=x.device)
    samples = [0]
    frame_every = max(1, n_steps // n_frames)

    def record(step, x, v, U, r2, f_over_r):
        T = eq.kinetic_temperature(v)
        out["E"].append((U + 0.5 * torch.einsum("sia,sia->s", v, v)) / N)  # energy per atom
        out["T"].append(T)
        out["P"].append(eq.virial_pressure(r2, f_over_r, T, rho, L))
        if step % sample_every == 0:
            counts.add_(eq.pair_histogram(r2, r_max, n_bins))
            samples[0] += 1
        if step % frame_every == 0:
            out["frames"].append(x.clone())
            out["frame_step"].append(step)
            out["g_running"].append(eq.radial_distribution(counts, samples[0], N, rho, r_max)[0])

    x, v = verlet(x, v, L, dt, n_steps, record=record, label="production")
    g, r = eq.radial_distribution(counts, samples[0], N, rho, r_max)
    return {
        "E": torch.stack(out["E"]),  # (K, S)
        "T": torch.stack(out["T"]),
        "P": torch.stack(out["P"]),
        "frames": torch.stack(out["frames"]),  # (F, S, N, 3)
        "frame_step": out["frame_step"],
        "g_running": torch.stack(out["g_running"]),  # (F, S, n_bins)
        "g": g,  # (S, n_bins)
        "r": r,  # (n_bins,)
        "x": x,
        "v": v,
    }


def first_peak(r: torch.Tensor, g: torch.Tensor) -> torch.Tensor:
    """Position of the highest point of g(r) for each state point.   (S,)"""
    return r[g.argmax(-1)]


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/physchem/lj_liquid_rdf/plot.py
"""Animate the production run: a thin slice of each box, g(r) building up, and the conserved energy."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif

from .tensors import NAMES

COLOURS = ["#16a34a", "#2563eb", "#9333ea"]
SLAB = 1.0  # σ: thickness of the slice drawn from each box


def animate(res: dict, L: torch.Tensor, rho: torch.Tensor, T: torch.Tensor, dt: float, path: Path) -> Path:
    """res['frames'] (F, S, N, 3); res['g_running'] (F, S, K) on res['r'] (K,); res['E'] (K_steps, S)."""
    frames = res["frames"].cpu().numpy()
    g_run, r = res["g_running"].cpu().numpy(), res["r"].cpu().numpy()
    E = res["E"].cpu().numpy()
    E_rel = (E - E[0]) * 1e4  # in units of 10⁻⁴ ε
    L, rho, T = L.cpu().numpy(), rho.cpu().numpy(), T.cpu().numpy()
    steps_f = res["frame_step"]
    t = np.arange(1, len(E) + 1) * dt
    S = frames.shape[1]

    fig = plt.figure(figsize=(9, 5.2))
    grid = fig.add_gridspec(2, 3, height_ratios=[1, 1])
    boxes = [fig.add_subplot(grid[0, s]) for s in range(S)]
    ax_g = fig.add_subplot(grid[1, :2])
    ax_e = fig.add_subplot(grid[1, 2])

    dots = []
    for s, ax in enumerate(boxes):
        dots.append(
            ax.scatter([], [], s=max(4.0, 900 / L[s] ** 2), color=COLOURS[s], edgecolors="#1f2937", linewidths=0.3)
        )
        ax.set_xlim(0, L[s])
        ax.set_ylim(0, L[s])
        ax.set_aspect("equal")
        ax.set_xticks([])
        ax.set_yticks([])
        ax.grid(False)
        ax.set_title(f"{NAMES[s]}: ρ* = {rho[s]:.2f}, T* = {T[s]:.2f}", fontsize=9)

    ax_g.axhline(1, color="#9ca3af", lw=1, ls="--")
    ax_g.axvline(2 ** (1 / 6), color="#f59e0b", lw=1, ls=":")
    ax_g.text(2 ** (1 / 6), 0.2, " $2^{1/6}\\sigma$", color="#b45309", fontsize=8)
    lines = [ax_g.plot([], [], color=COLOURS[s], lw=1.6, label=NAMES[s])[0] for s in range(S)]
    ax_g.set_xlim(0, r[-1])
    ax_g.set_ylim(0, 1.1 * max(np.nanmax(g_run[-1]), 3.0))
    ax_g.set_xlabel("pair distance r (σ)")
    ax_g.set_ylabel("g(r)")
    ax_g.legend(fontsize=8)
    title_g = ax_g.set_title("", fontsize=10)

    spread = max(np.abs(E_rel).max(), 1.0)
    ax_e.set_xlim(0, t[-1])
    ax_e.set_ylim(-1.5 * spread, 1.5 * spread)
    elines = [ax_e.plot([], [], color=COLOURS[s], lw=1.2)[0] for s in range(S)]
    ax_e.set_xlabel("time (τ)")
    ax_e.set_ylabel("E/N − E/N at start (10⁻⁴ ε)")
    ax_e.set_title("NVE: energy is conserved", fontsize=10)

    def update(f: int):
        for s in range(S):
            x = frames[f, s]
            slab = np.abs(x[:, 2] - 0.5 * L[s]) < 0.5 * SLAB
            dots[s].set_offsets(x[slab, :2])
            lines[s].set_data(r, g_run[f, s])
        k = steps_f[f] + 1
        for s in range(S):
            elines[s].set_data(t[:k], E_rel[:k, s])
        title_g.set_text(f"radial distribution function, averaged up to t = {k * dt:.1f} τ")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(steps_f), 40), path, fps=6)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'rho_liquid': 0.84,
    'T_liquid': 0.72,
    'n_cells': 5,
    'dt': 0.005,
    'equilibration_steps': 600,
    'production_steps': 700
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.physchem.lj_liquid_rdf")]:
    del sys.modules[name]
main = importlib.import_module("problems.physchem.lj_liquid_rdf.main")
problem_dir = Path("problems/physchem/lj_liquid_rdf")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
